# 1. 16-720 PA1: Local Features and Object-Locked Video

**Build a classical local-feature pipeline** — Harris corners, a compact gradient descriptor, ratio-test matching — **then use it to make a video in which your chosen object stays locked in place while the scene moves around it.**

| Release | Due | Base points | Extra credit (cap) | Expected effort | Platform |
|---|---|---|---|---|---|
| Sep. 8, 2026 | Sep. 29, 2026 | 100 | 10 | 8–10 hours | Google Colab (CPU is sufficient) |

`detect` → `describe` → `match` → `estimate` → `warp` → 🎥 object-locked video

## 1.1. Objectives and workflow

You will:

1. **Derive** local image structure by hand (Part A, in the PDF).
2. **Implement** the six graded functions in `student/pa1.py` (Part B).
3. **Stress-test** the representation's invariances (Part C).
4. **Create** the 10–20 second object-locked Hero Result video (Part D).

Work top to bottom. Complete the six functions first, then run the whole notebook from a clean runtime. Write generated files only under `artifacts/`.

> 💡 Colab shows a clickable table of contents for this notebook: **View → Table of contents**.

## 1.2. Academic-integrity and AI boundary

> ⚠️ **Generative AI is NOT permitted** for Part A or for the required Part B implementations in `student/pa1.py` — not to generate, complete, translate, explain, debug, or rewrite them. It **is permitted** for Parts C, D, and E **with disclosure and comprehension**.

Supplied helper code is allowed everywhere. Do not call OpenCV feature detectors, descriptors, or matchers in the six graded functions.

# 2. Environment, seed, and data

**In a fresh Colab runtime:**

1. Run the next cell.
2. When the upload picker appears, select the **complete, unopened repository ZIP** downloaded from GitHub. Uploading only this notebook is not sufficient.
3. The cell verifies and extracts the repository, installs `requirements-colab.txt`, selects the repository root, seeds everything with `16720`, and generates the deterministic data assets.
4. If Colab asks to restart after installation: restart, then run the cell again — it reuses the extracted repository.

In [ ]:
# SUPPLIED: locate a complete repository or upload its ZIP in Colab.
from pathlib import Path
import io
import os
import subprocess
import sys
import zipfile

REPOSITORY_MARKERS = (
    "requirements-colab.txt",
    "student/pa1.py",
    "common/pa1_utils.py",
    "data/setup_assets.py",
)
KNOWN_EXTRACT_ROOT = Path(os.environ.get(
    "CV16720_REPOSITORY_EXTRACT_ROOT", "/content/cv16720_pa1_repository"
)).resolve()

def _is_repository_root(candidate):
    return all((candidate / marker).is_file() for marker in REPOSITORY_MARKERS)

def _single_repository_beneath(base):
    if not base.is_dir():
        return None
    if _is_repository_root(base):
        return base.resolve()
    roots = sorted(
        path.resolve() for path in base.iterdir()
        if path.is_dir() and _is_repository_root(path)
    )
    if len(roots) > 1:
        raise RuntimeError(
            f"Expected at most one PA1 repository beneath {base}; "
            f"found {len(roots)}."
        )
    return roots[0] if roots else None

def _find_repository_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if _is_repository_root(candidate):
            return candidate
    return _single_repository_beneath(KNOWN_EXTRACT_ROOT)

ROOT = _find_repository_root()
if ROOT is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Could not find the PA1 repository. Clone or extract the complete "
            "repository and run this notebook from its root."
        ) from exc

    print("Upload the complete, unopened repository ZIP downloaded from GitHub.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError("Upload exactly one repository ZIP, then rerun this cell.")

    extract_root = KNOWN_EXTRACT_ROOT
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            target = (extract_root / member_path).resolve()
            mode = (member.external_attr >> 16) & 0o170000
            if (member_path.is_absolute() or ".." in member_path.parts
                    or (target != extract_root and extract_root not in target.parents)
                    or mode == 0o120000):
                raise RuntimeError(f"Unsafe ZIP member rejected: {member.filename!r}")
        archive.extractall(extract_root)

    ROOT = _single_repository_beneath(extract_root)
    if ROOT is None:
        raise RuntimeError(
            "The uploaded ZIP did not contain a complete top-level PA1 repository."
        )

os.chdir(ROOT)
requirements = ROOT / "requirements-colab.txt"
if "google.colab" in sys.modules and os.environ.get("CV16720_PA1_COLAB_ROOT") != str(ROOT):
    if not requirements.is_file():
        raise RuntimeError(f"Missing {requirements}; upload the complete repository ZIP.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)
    ])
    os.environ["CV16720_PA1_COLAB_ROOT"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from student import pa1
from common import pa1_utils as utils

rng = utils.set_seed(16720)
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
ASSETS = ROOT / "data" / "generated"
subprocess.run([
    sys.executable, str(ROOT / "data" / "setup_assets.py"),
    "--output", str(ASSETS),
], check=True)
print("Python:", sys.version.split()[0], "| repository:", ROOT)


## 2.1. Fixed conventions

- Images are `(H, W)` or `(H, W, C)` arrays; loaded grayscale images are `float32` in `[0, 1]`.
- Points are Cartesian `(x, y)`: an integer keypoint `[x, y]` indexes an image as `image[y, x]`.
- `Ix` is zero in the first/last **columns**; `Iy` is zero in the first/last **rows**.
- The NMS threshold is relative to the largest positive response.
- Descriptor cells are row-major; the orientation bin is the fastest-changing index.
- Geometry arrays are `float64`; the supplied estimator maps **current-frame points to first-frame reference points**.

# 3. B1: Read the pipeline (5 points)

**Answer all five questions in the written PDF before implementing.** They are quoted verbatim from Section 7.1 of the assignment handout; inspect `utils.gaussian_blur` and `utils.estimate_affine_partial` in `common/pa1_utils.py` while answering.

1. A grayscale image has shape `(480, 640)`. What are the shapes of `Ix` and `Iy`, and at which array entry is keypoint `[137, 82]` accessed?
2. Inspect the supplied separable Gaussian aggregation helper. Why does its output preserve `(H, W)`, and how are samples beyond the image boundary defined?
3. With `patch_size=16`, `cells=4`, and `bins=8`, state the exact patch bounds around integer keypoint `(x, y)`, the spatial size of each cell, and the descriptor length.
4. What does L2 normalization do to the descriptor under a global positive contrast scaling of the image? What descriptor is produced by a flat patch?
5. The matcher receives reference descriptors as its first argument and current-frame descriptors as its second. Given `matches`, which reference/current keypoint arrays are passed to the supplied estimator, and in which direction does the estimated transform map points?

# 4. B2: Harris detector (12 points)

Implement in `student/pa1.py`: `compute_image_gradients`, `compute_second_moment`, `compute_corner_response`, `nonmax_suppression`. Then run the two cells below.

> ✅ **Checkpoint:** the white square yields **exactly 4 keypoints**, one per corner; the flat image yields **0** points and zero max response; a larger NMS radius never returns more points than a smaller one.

In [ ]:
square_rgb = utils.load_image(ASSETS / 'square.png')
square = utils.to_gray_float(square_rgb)
Ix, Iy = pa1.compute_image_gradients(square)
Sxx, Sxy, Syy = pa1.compute_second_moment(Ix, Iy, sigma=1.5)
R = pa1.compute_corner_response(Sxx, Sxy, Syy, k=0.04)
keypoints, scores = pa1.nonmax_suppression(R, threshold=0.01, radius=8, max_points=500)
print('square points:', keypoints.shape, '| expected after completion: 4')
utils.plot_gradients_and_response(square, Ix, Iy, R, keypoints);


In [ ]:
flat = utils.to_gray_float(utils.load_image(ASSETS / 'flat.png'))
fx, fy = pa1.compute_image_gradients(flat)
flat_R = pa1.compute_corner_response(*pa1.compute_second_moment(fx, fy, 1.5), 0.04)
flat_points, _ = pa1.nonmax_suppression(flat_R, 0.01, 8, 500)
print('flat max response:', float(flat_R.max()), '| points:', len(flat_points), '| expected: 0 and 0')

photo_rgb = utils.load_image(ASSETS / 'photo_texture.png')
photo = utils.to_gray_float(photo_rgb)
px, py = pa1.compute_image_gradients(photo)
photo_R = pa1.compute_corner_response(*pa1.compute_second_moment(px, py, 1.5), 0.04)
small_points, small_scores = pa1.nonmax_suppression(photo_R, 0.01, 4, 500)
large_points, large_scores = pa1.nonmax_suppression(photo_R, 0.01, 16, 500)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
utils.plot_keypoints(photo_rgb, small_points, small_scores, f'radius 4: {len(small_points)}', axes[0])
utils.plot_keypoints(photo_rgb, large_points, large_scores, f'radius 16: {len(large_points)}', axes[1]);

# 5. B3: Compact gradient descriptor (10 points)

Implement `describe_keypoints` with `patch_size=16`, `cells=4`, and `bins=8`. Produce complete diagnostic figures for two keypoints, and for one selected keypoint compare the original descriptor against one explicitly parameterized perturbation, annotating dominant cells/orientations and changes.

> ✅ **Checkpoint:** every descriptor has **128 entries**; non-flat descriptors have L2 norm just below or numerically equal to **1**; flat patches stay **exactly zero**.

In [ ]:
descriptors = pa1.describe_keypoints(Ix, Iy, keypoints, 16, 4, 8)
print('descriptor shape:', descriptors.shape, '| first norms:', np.linalg.norm(descriptors[:4], axis=1))
for index in range(min(2, len(keypoints))):
    utils.plot_descriptor_diagnostics(square, Ix, Iy, keypoints[index], descriptors[index]);
# TODO(student): add the required perturbed-patch descriptor comparison here: pick one
# keypoint, apply one parameterized perturbation from utils, and annotate dominant
# cells/orientations and their changes for the PDF.

# 6. B4: Feature matching (8 points)

Implement `match_descriptors` — a Euclidean nearest-neighbor ratio test with default ratio `0.80` — then run the cell below on the supplied easy pair.

> ✅ **Checkpoint:** match rows and confidences have equal length and are ordered by decreasing confidence; the easy pair yields enough geometrically consistent correspondences for the supplied estimator. Repetitive texture can still push through a bundle of low-confidence many-to-one matches — exactly the ambiguity the PDF question asks you to analyze.

In [ ]:
def detect_and_describe(image_rgb, radius=8, max_points=500):
    gray = utils.to_gray_float(image_rgb)
    gx, gy = pa1.compute_image_gradients(gray)
    response = pa1.compute_corner_response(*pa1.compute_second_moment(gx, gy, 1.5), 0.04)
    points, point_scores = pa1.nonmax_suppression(response, 0.01, radius, max_points)
    vectors = pa1.describe_keypoints(gx, gy, points, 16, 4, 8)
    return points, vectors

reference_rgb = utils.load_image(ASSETS / 'texture.png')
easy_rgb = utils.load_image(ASSETS / 'match_easy.png')
reference_points, reference_desc = detect_and_describe(reference_rgb)
easy_points, easy_desc = detect_and_describe(easy_rgb)
matches, confidence = pa1.match_descriptors(reference_desc, easy_desc, 0.8)
print('features:', len(reference_points), len(easy_points), '| matches:', len(matches))
utils.plot_matches(reference_rgb, easy_rgb, reference_points, easy_points, matches, confidence);
# TODO(student): in the PDF, identify one correct match and one incorrect or rejected
# ambiguous match from this figure.

# 7. Part C: Invariance experiments (15 points)

**C1 (8 points) — four conditions, one shared reference view that YOU capture** (assignment handout, Section 8.1):

| # | Condition | Source |
|---|---|---|
| 1 | Real illumination change | your camera, same scene re-photographed |
| 2 | Real scale and/or viewpoint change | your camera, same scene re-photographed |
| 3 | Synthetic Gaussian blur | supplied transform applied to your reference photo |
| 4 | Synthetic in-plane rotation | supplied transform applied to your reference photo |

Every row reports the same five metrics — reference features, destination features, ratio-passed matches, RANSAC inliers, inlier fraction — and needs a readable match visualization. The next cell pre-wires the two synthetic conditions on course assets as a **format demo only**; its TODO explains how to build all four graded rows against your own reference.

**C2 (4 points):** two prediction/result/theory analyses; at least one must analyze **descriptor invariance**. **C3 (3 points):** one diagnosed tracker failure. Both go in the PDF. GenAI is permitted in Part C with disclosure.

In [ ]:
# SUPPLIED: shared metric helper. Every C1 row must use it unchanged so rows stay comparable.
def pair_metrics(reference_rgb, changed_rgb):
    p1, d1 = detect_and_describe(reference_rgb)
    p2, d2 = detect_and_describe(changed_rgb)
    pair_matches, _ = pa1.match_descriptors(d1, d2, 0.8)
    transform, inliers, info = utils.estimate_affine_partial(
        p2[pair_matches[:, 1]] if len(pair_matches) else np.empty((0, 2)),
        p1[pair_matches[:, 0]] if len(pair_matches) else np.empty((0, 2)),
    )
    return {'reference_features': len(p1), 'changed_features': len(p2),
            'matches': len(pair_matches), 'inliers': int(inliers.sum()),
            'inlier_fraction': float(info['ratio'])}

results = {}

# SUPPLIED demo: the two required synthetic conditions applied to a course asset.
# These rows demonstrate the metric format only; they are NOT your graded C1 rows.
synthetic_blur = utils.apply_gaussian_blur(reference_rgb.astype(np.float32) / 255.0, 2.0)
synthetic_rotation, _ = utils.rotate_and_scale(reference_rgb, 20.0, 1.0)
results['demo_synthetic_blur_sigma_2.0'] = pair_metrics(reference_rgb, synthetic_blur)
results['demo_synthetic_rotation_20deg'] = pair_metrics(reference_rgb, synthetic_rotation)

# TODO(student): build your graded C1 table. All FOUR rows share ONE reference view
# that you capture yourself (assignment handout, Section 8.1).
#   1. Photograph ONE textured reference view, then re-photograph the same scene under
#      (a) a real illumination change (for example lamp on versus off) and
#      (b) a real scale and/or viewpoint change (for example step back, or rotate ~30 degrees).
#      Record the physical setup of each condition for the PDF.
#   2. Upload the three photos in Colab (folder icon -> upload), then load them:
#        my_reference = utils.load_image('/content/reference.jpg')
#        my_illumination = utils.load_image('/content/illumination.jpg')
#        my_viewpoint = utils.load_image('/content/viewpoint.jpg')
#      Downscale large phone photos first so feature counts stay comparable to the
#      supplied 480x360 assets, for example:
#        import cv2; image = cv2.resize(image, None, fx=0.15, fy=0.15)  # 4032x3024 -> ~605x454
#   3. Produce all four graded rows against YOUR reference with the SAME helper:
#        results['real_illumination'] = pair_metrics(my_reference, my_illumination)
#        results['real_scale_viewpoint'] = pair_metrics(my_reference, my_viewpoint)
#        results['synthetic_blur_sigma_2.0'] = pair_metrics(my_reference,
#            utils.apply_gaussian_blur(my_reference.astype(np.float32) / 255.0, 2.0))
#        results['synthetic_rotation_20deg'] = pair_metrics(my_reference,
#            utils.rotate_and_scale(my_reference, 20.0, 1.0)[0])

# TODO(student): for EVERY graded row, also show a readable match/inlier visualization
# (reuse utils.plot_matches as in the B4 cell) and copy the final four-row table into the PDF.

for condition, row in results.items():
    print(condition, '->', row)

# 8. Part D: Object-locked video (25 points)

**Capture** a safe 10–20 second clip, or use a staff-approved alternative for access, privacy, safety, or accessibility reasons. Choose a **textured, matte, approximately rigid** target that stays visible while the camera moves.

**Pipeline** (supplied harness): downsample to at most width 640 and 15 fps → set a first-frame `(x, y, width, height)` ROI around the target → the harness matches ROI reference descriptors to each current frame, estimates current-to-reference affine-partial motion, and holds a failed transform for at most five frames.

> 🎬 **Deliverable:** an H.264/yuv420p MP4 — at most 720p / 30 fps, 10–20 seconds, at most 50 MB.

In [ ]:
VIDEO_PATH = ASSETS / 'smoke_object_motion.mp4'  # TODO(student): replace with your own capture.
frames, fps = utils.read_video(VIDEO_PATH, target_fps=15, max_width=640)
plt.figure(figsize=(9, 5)); plt.imshow(frames[0]); plt.grid(); plt.title('Read x/y axes and set ROI below');
TARGET_ROI = (120, 90, 240, 180)  # TODO(student): replace after inspecting your first frame.
stabilized, diagnostics = utils.object_lock_frames(
    frames, detect_and_describe, pa1.match_descriptors, TARGET_ROI, ratio=0.8
)
hero_frames = utils.side_by_side_frames(frames, stabilized)
hero_path, codec = utils.write_video(ARTIFACTS / 'pa1_hero.mp4', hero_frames, fps)
print(hero_path, codec, diagnostics[:3])
final_path = utils.transcode_h264(hero_path, ARTIFACTS / 'pa1_hero_h264.mp4')
print('submission candidate:', final_path, '| verify duration and size before submission')
# The final MP4 must be H.264/yuv420p, <=720p/30 fps, 10-20 seconds, and <=50 MB.

# 9. Submission audit

Run this final self-check before submitting — every box must be true:

- [ ] `student/pa1.py` contains your six implementations and no prohibited black boxes.
- [ ] The notebook runs top to bottom with deterministic seed 16720.
- [ ] The LaTeX PDF contains A1-D4 answers and every graded figure/table.
- [ ] The H.264/yuv420p Hero Result is 10-20 seconds and at most 50 MB.
- [ ] Collaborators, conventional sources, data, and permitted AI assistance are acknowledged.
- [ ] D4 is 150-250 words. If no GenAI was used, it says so and explains your most important independently implemented component and validation process.
- [ ] Optional showcase participation is affirmative and separate from submission; all extra credit remains capped at 10.